# GP2: MelGAN Neural Vocoder

**Training time: ~1h 40min on T4** (40 epochs + AMP)

> Runtime → Change runtime type → **T4 GPU** before running!

In [ ]:
## 1. Install
!pip install git+https://github.com/idiap/coqui-ai-TTS.git -q
!pip install torchaudio soundfile matplotlib -q
print('Done!')

In [ ]:
## 2. Mount Drive & Config
from google.colab import drive
drive.mount('/content/drive')
import os

CKPT_DIR = '/content/drive/MyDrive/gp2_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

# Hyperparameters — 40 epochs ~ 1h 40min on T4 with AMP
EPOCHS, BATCH_SIZE, LR = 40, 32, 1e-4
SEGMENT_FRAMES = 32   # 32 * 256 = 8192 audio samples per item
SAVE_EVERY, VAL_EVERY = 10, 5
DEVICE = 'cuda'

# Mel config — must match FastPitch (Coqui-TTS) AudioProcessor
SAMPLE_RATE, N_FFT, WIN_LENGTH, HOP_LENGTH = 22050, 1024, 1024, 256
N_MELS, F_MIN, F_MAX, MEL_POWER, LOG_EPS = 80, 0.0, 8000.0, 1.5, 1e-5

print(f'Config: {EPOCHS} epochs | batch={BATCH_SIZE} | lr={LR}')
print(f'Checkpoints: {CKPT_DIR}')

In [ ]:
## 3. Imports
import torch, random, time
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast, GradScaler

device = torch.device(DEVICE)
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
## 4. Model Architecture

class ResBlock(nn.Module):
    def __init__(self, ch, dilations=(1, 3, 9)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(ch, ch, 3, padding=d, dilation=d),
                nn.LeakyReLU(0.2),
                nn.Conv1d(ch, ch, 3, padding=1),
            ) for d in dilations
        ])
    def forward(self, x):
        for c in self.convs: x = x + c(x)
        return x

class UpsampleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.up  = nn.ConvTranspose1d(in_ch, out_ch, stride*2, stride=stride, padding=stride//2)
        self.res = ResBlock(out_ch)
    def forward(self, x):
        return self.res(self.up(F.leaky_relu(x, 0.2)))

class Generator(nn.Module):
    # MelGAN: mel [B,80,T] -> waveform [B,1,256*T]
    def __init__(self, n_mels=80):
        super().__init__()
        self.pre  = nn.Conv1d(n_mels, 512, 7, padding=3)
        self.ups  = nn.Sequential(
            UpsampleBlock(512, 256, 8),
            UpsampleBlock(256, 128, 8),
            UpsampleBlock(128,  64, 4),
        )
        self.post = nn.Sequential(
            nn.LeakyReLU(0.2),
            nn.Conv1d(64, 1, 7, padding=3),
            nn.Tanh(),
        )
    def forward(self, x):
        return self.post(self.ups(self.pre(x)))

class SubDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Conv1d(1,    16,  15, 1,  padding=7),
            nn.Conv1d(16,   64,  41, 4,  padding=20, groups=4),
            nn.Conv1d(64,  256,  41, 4,  padding=20, groups=16),
            nn.Conv1d(256, 1024, 41, 4,  padding=20, groups=64),
            nn.Conv1d(1024,1024,  5, 1,  padding=2),
            nn.Conv1d(1024,   1,  3, 1,  padding=1),
        ])
    def forward(self, x):
        feats = []
        for layer in self.layers[:-1]:
            x = F.leaky_relu(layer(x), 0.2)
            feats.append(x)
        x = self.layers[-1](x)
        feats.append(x)
        return x, feats

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discs = nn.ModuleList([SubDiscriminator() for _ in range(3)])
        self.pools = nn.ModuleList([nn.Identity(), nn.AvgPool1d(2,2), nn.AvgPool1d(4,4)])
    def forward(self, x):
        outs, feats = [], []
        for pool, disc in zip(self.pools, self.discs):
            o, f = disc(pool(x))
            outs.append(o); feats.append(f)
        return outs, feats

G = Generator().to(device)
D = MultiScaleDiscriminator().to(device)
print(f'Generator:     {sum(p.numel() for p in G.parameters()):,} params')
print(f'Discriminator: {sum(p.numel() for p in D.parameters()):,} params')

In [ ]:
## 5. Dataset (LJSpeech)

_MEL_FB = {}

def get_mel_fb(dev):
    k = str(dev)
    if k not in _MEL_FB:
        _MEL_FB[k] = torchaudio.functional.melscale_fbanks(
            n_freqs=N_FFT//2+1, f_min=F_MIN, f_max=F_MAX,
            n_mels=N_MELS, sample_rate=SAMPLE_RATE, mel_scale='htk',
        ).to(dev)
    return _MEL_FB[k]

def compute_log_mel(wav):
    window = torch.hann_window(WIN_LENGTH, device=wav.device)
    spec = torch.stft(wav, N_FFT, HOP_LENGTH, WIN_LENGTH, window,
                      center=True, return_complex=True)
    mel = torch.matmul(spec.abs().pow(MEL_POWER).T, get_mel_fb(wav.device)).T
    return torch.log(mel.clamp(min=LOG_EPS))

class LJSpeechDataset(Dataset):
    def __init__(self, root):
        torchaudio.datasets.LJSPEECH(root, download=True)
        wav_dir  = os.path.join(root, 'LJSpeech-1.1', 'wavs')
        seg_samp = SEGMENT_FRAMES * HOP_LENGTH
        self.seg = seg_samp
        self.paths = sorted([
            os.path.join(wav_dir, fn) for fn in os.listdir(wav_dir)
            if fn.endswith('.wav')
            and sf.info(os.path.join(wav_dir, fn)).frames >= seg_samp + WIN_LENGTH
        ])
        print(f'Dataset: {len(self.paths)} files')
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        data, sr = sf.read(self.paths[idx], dtype='float32', always_2d=False)
        wav = torch.from_numpy(data)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        start = random.randint(0, max(0, wav.shape[0] - self.seg))
        audio = wav[start:start+self.seg]
        if audio.shape[0] < self.seg:
            audio = F.pad(audio, (0, self.seg - audio.shape[0]))
        return compute_log_mel(audio)[:, :SEGMENT_FRAMES], audio

ds = LJSpeechDataset('/content/data')
n_val = min(500, len(ds)//10)
train_ds, val_ds = random_split(ds, [len(ds)-n_val, n_val],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=True)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
## 6. Training (LSGAN + Feature Matching + L1, AMP)

def disc_loss(r_outs, f_outs):
    return sum(torch.mean((r-1)**2) + torch.mean(f**2) for r, f in zip(r_outs, f_outs))

def gen_adv_loss(f_outs):
    return sum(torch.mean((f-1)**2) for f in f_outs)

def feat_loss(r_all, f_all):
    loss = 0.0
    for rf, ff in zip(r_all, f_all):
        for ri, fi in zip(rf[:-1], ff[:-1]):
            loss += F.l1_loss(fi, ri.detach())
    return loss

g_opt    = torch.optim.Adam(G.parameters(), LR, betas=(0.5, 0.9))
d_opt    = torch.optim.Adam(D.parameters(), LR, betas=(0.5, 0.9))
g_sched  = torch.optim.lr_scheduler.StepLR(g_opt, step_size=100, gamma=0.5)
d_sched  = torch.optim.lr_scheduler.StepLR(d_opt, step_size=100, gamma=0.5)
g_scaler = GradScaler('cuda')
d_scaler = GradScaler('cuda')

g_losses, d_losses, val_l1s = [], [], []
t_start = time.time()

for epoch in range(1, EPOCHS+1):
    G.train(); D.train()
    t0 = time.time(); g_sum = d_sum = 0.0

    for mel, audio in train_loader:
        mel  = mel.to(device)
        real = audio.unsqueeze(1).to(device)
        T    = real.shape[-1]

        with autocast('cuda'):
            fake = G(mel)[..., :T]

        # Discriminator step
        d_opt.zero_grad()
        with autocast('cuda'):
            r_outs, _ = D(real)
            f_outs, _ = D(fake.detach())
            ld = disc_loss(r_outs, f_outs)
        d_scaler.scale(ld).backward()
        d_scaler.step(d_opt); d_scaler.update()

        # Generator step
        g_opt.zero_grad()
        with autocast('cuda'):
            f_outs, f_feats = D(fake)
            _, r_feats      = D(real)
            lg = gen_adv_loss(f_outs) + 10.0*feat_loss(r_feats, f_feats) + 10.0*F.l1_loss(fake, real)
        g_scaler.scale(lg).backward()
        g_scaler.step(g_opt); g_scaler.update()

        g_sum += lg.item(); d_sum += ld.item()

    g_sched.step(); d_sched.step()
    nb = len(train_loader)
    g_losses.append(g_sum/nb); d_losses.append(d_sum/nb)
    elapsed = time.time()-t0
    total   = time.time()-t_start
    print(f'Epoch {epoch:03d}/{EPOCHS} | G={g_sum/nb:.4f} D={d_sum/nb:.4f} | {elapsed:.0f}s | {total/60:.0f}min')

    if epoch % VAL_EVERY == 0:
        G.eval(); vl = 0.0
        with torch.no_grad():
            for mel, audio in val_loader:
                mel  = mel.to(device)
                real = audio.unsqueeze(1).to(device)
                vl  += F.l1_loss(G(mel)[..., :real.shape[-1]], real).item()
        val_l1s.append(vl/len(val_loader))
        print(f'  -> val L1 = {val_l1s[-1]:.4f}')

    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        path = os.path.join(CKPT_DIR, f'vocoder_epoch{epoch:03d}.pt')
        torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'epoch': epoch}, path)
        print(f'  Saved: {path}')

print(f'Training complete! {(time.time()-t_start)/60:.0f} min total')

In [ ]:
## 7. Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(g_losses, label='G loss')
ax1.plot(d_losses, label='D loss')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Training Losses'); ax1.legend()
epochs_val = list(range(VAL_EVERY, EPOCHS+1, VAL_EVERY))[:len(val_l1s)]
ax2.plot(epochs_val, val_l1s, 'o-', color='green')
ax2.set(xlabel='Epoch', ylabel='L1', title='Validation L1')
plt.tight_layout()
plot_path = os.path.join(CKPT_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150); plt.show()
print(f'Saved: {plot_path}')

In [ ]:
## 8. Load Best Checkpoint
ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')])
print('Checkpoints:', ckpts)

best_ckpt = os.path.join(CKPT_DIR, ckpts[-1])
G_inf = Generator().to(device)
ckpt  = torch.load(best_ckpt, map_location=device)
G_inf.load_state_dict(ckpt['G'])
G_inf.eval()
print(f'Loaded: {ckpts[-1]} | epoch', ckpt['epoch'])

In [ ]:
## 9. Load FastPitch (Text -> Mel)
from TTS.api import TTS
from TTS.tts.utils.synthesis import synthesis

class TextToSpecConverter:
    def __init__(self, model_name='tts_models/en/ljspeech/fast_pitch'):
        self.tts      = TTS(model_name=model_name)
        self.model    = self.tts.synthesizer.tts_model
        self.config   = self.tts.synthesizer.tts_config
        self.use_cuda = torch.cuda.is_available()
        print(f'FastPitch loaded | cuda={self.use_cuda}')
    def text2spec(self, text):
        out = synthesis(self.model, text, self.config, self.use_cuda,
                        use_griffin_lim=False, do_trim_silence=False)
        mel = out['outputs']['model_outputs'][0].detach().cpu().numpy()
        mel = self.model.ap.denormalize(mel.T).T
        return mel  # [80, T] log-mel

t2s = TextToSpecConverter()

In [ ]:
## 10. Generate 5 Test Audio Samples
import IPython.display as ipd

SENTENCES = [
    'Tourists found the octagonal lighthouse after a two-mile hike through fog',
    'In twenty forty-nine, neural implants became standard for city workers',
    'He whispered something in Icelandic I could not quite translate',
    'Lucia balanced a porcelain vase on her elbow without flinching',
    'They gathered quietly beneath the turbine as the wind picked up speed',
]

audio_dir = os.path.join(CKPT_DIR, 'audio_samples')
os.makedirs(audio_dir, exist_ok=True)

for i, text in enumerate(SENTENCES, 1):
    print(f'[{i}] {text}')
    mel_np = t2s.text2spec(text)
    mel    = torch.tensor(mel_np, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        wav = G_inf(mel).squeeze(0).clamp(-1, 1).cpu()
    path = os.path.join(audio_dir, f'sample_{i:02d}.wav')
    torchaudio.save(path, wav, SAMPLE_RATE)
    print(f'  -> {path}')
    ipd.display(ipd.Audio(wav.numpy(), rate=SAMPLE_RATE))

print(f'All 5 samples saved to: {audio_dir}')